# 02 · Baseline vs improved (and the CNN path)

Reads results written by `python -m visionguard.cli pipeline`.

In [1]:
import json

import pandas as pd

from visionguard.common.config import get_settings
from visionguard.storage.database import Database

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
db = Database(get_settings().db_path)
m = db.query_df("SELECT name, kind, n_features, size_kb, latency_ms_p95, metrics_json FROM models")
rows = []
for r in m.itertuples():
    mm = json.loads(r.metrics_json)
    rows.append({"model": r.name, "macro_f1": mm["macro_f1"], "ci95": mm["macro_f1_ci95"], "defect_recall": mm["defect_recall"],
                 "false_alarms": mm["false_alarm_rate"], "p95_ms": r.latency_ms_p95, "size_kb": r.size_kb})
pd.DataFrame(rows).round(3)

,model,macro_f1,ci95,defect_recall,false_alarms,p95_ms,size_kb
0,baseline,0.651,"[0.584206200619005, 0.704903374830886]",0.929,0.205,3.636,3.280
1,improved,0.850,"[0.8022425966236907, 0.8940111583229692]",0.909,0.075,8.142,1004.569
2,robust,0.825,"[0.7751005660973079, 0.870919947798765]",0.864,0.062,12.444,1547.726


## Per-class recall

In [2]:
pd.DataFrame({r.name: {c: v["recall"] for c, v in json.loads(r.metrics_json)["per_class"].items()} for r in m.itertuples()}).round(3)

,baseline,improved,robust
normal,0.795,0.925,0.938
scratch,0.545,0.636,0.568
inclusion,0.512,0.878,0.854
crack,0.725,0.925,0.875
patch,0.828,0.897,0.828


## CNN transfer learning
`src/visionguard/models/cnn.py` implements ResNet18 (ImageNet weights, 1-channel stem) and a SmallCNN;
`training/train_cnn.py` freezes the backbone for 3 epochs, then fine-tunes with class-weighted loss and
early stopping on validation macro-F1. Run it with:

```bash
pip install -r requirements-cnn.txt
python -m visionguard.cli train-cnn
```

CNN numbers are deliberately **not** reported until that run has been executed on your hardware.